# Faking a Kernel With 400 Random Frequencies

**Topic:** Random Fourier features &nbsp;·&nbsp; **Difficulty:** Hard

Kernel methods die at scale: the Gram matrix for 100,000 points has ten billion
entries. Random Fourier features dodge this entirely: draw random frequencies
from the kernel's Fourier transform, project, and the *inner product* of the
resulting features approximates the kernel.

The scaling constant is where everyone slips. Get `sqrt(2/D)` wrong and the
approximation is off by a constant factor that looks like a bug in your kernel.

## The data (created in the next cell)

- `X` `(500, 6)`: points to compare kernels on.
- `gamma = 0.3`, feature count `D = 400`.
- Use `np.random.default_rng(31337)` for the random frequencies and phases.

## Your task

With **NumPy only**:

1. Draw `W` of shape `(6, D)` from `N(0, 2*gamma)`, that is, standard normal
   scaled by `sqrt(2 * gamma)`, and phases `b` uniform on `[0, 2*pi)`.
2. Build features `Z = sqrt(2 / D) * cos(X @ W + b)`, shape `(500, 400)`.
3. `K_exact`: the true RBF Gram matrix `exp(-gamma * ||x_i - x_j||^2)`.
4. `K_approx = Z @ Z.T`.
5. Set `answers` to a dict with `"Z_shape_ok"` (`1.0` if `Z` is `(500, 400)`),
   `"max_abs_err"`, `"mean_abs_err"` (between the two Gram matrices), and
   `"diag_mean"`: the mean of `K_approx`'s diagonal, which should be near 1.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks the mean absolute error is below 0.05 and the diagonal is near 1.

In [ ]:
import numpy as np

rng = np.random.default_rng(1313)
X = rng.normal(0, 1.0, (500, 6))
gamma = 0.3
D = 400
print(f"X {X.shape}, gamma = {gamma}, D = {D}")
print(f"exact Gram would be {X.shape[0]}x{X.shape[0]}, and features are {X.shape[0]}x{D}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("Z_shape_ok", [], 1.0, -0.16743801481449916, 1e-06),
    ("max_abs_err", [], 0.2095573641310618, -0.03508786903986412, 1e-06),
    ("mean_abs_err", [], 0.0369073770220399, -0.006179697940580623, 1e-06),
    ("diag_mean", [], 0.9982651939777759, -0.1671475423380497, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "mean_abs_err" in answers and "diag_mean" in answers:
    _good = answers["mean_abs_err"] < 0.05 and abs(answers["diag_mean"] - 1.0) < 0.1
    print(f"  {'approximation is close':<24} {'ok' if _good else 'wrong'}  "
          f"(mean err {answers['mean_abs_err']:.4f}, diag {answers['diag_mean']:.3f})")
    _ok = _ok and _good

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

Bochner's theorem says a shift-invariant kernel is the Fourier transform of a
probability measure. For the RBF that measure is Gaussian with variance
`2*gamma`. Sampling `D` frequencies and using the `cos(w.x + b)` trick gives an
unbiased estimate of the kernel, and the `sqrt(2/D)` normalization is what makes
`E[z(x).z(y)] = k(x, y)` rather than `D/2` times it. Error falls as
`1/sqrt(D)`, so quadrupling the features halves the error.

```python
import numpy as np

rng2 = np.random.default_rng(31337)
d = X.shape[1]

W = rng2.normal(0, 1.0, (d, D)) * np.sqrt(2 * gamma)
b = rng2.uniform(0, 2 * np.pi, D)

Z = np.sqrt(2.0 / D) * np.cos(X @ W + b)

sq = (np.sum(X ** 2, 1)[:, None] + np.sum(X ** 2, 1)[None, :] - 2 * X @ X.T)
K_exact = np.exp(-gamma * np.maximum(sq, 0.0))
K_approx = Z @ Z.T

err = np.abs(K_exact - K_approx)
answers = {
    "Z_shape_ok": float(Z.shape == (X.shape[0], D)),
    "max_abs_err": float(err.max()),
    "mean_abs_err": float(err.mean()),
    "diag_mean": float(np.diag(K_approx).mean()),
}
```